In [ ]:
# ==========================================
# SETUP & MOCK DATA (Context)
# ==========================================
# NOTE: This cell mimics the output of "Task 1: Data Management".
# We assume this data has already been ingested, cleaned, and indexed by time.

import pandas as pd
import numpy as np
from dataclasses import dataclass
from typing import List, Literal, Optional

# Generating synthetic OHLCV data for AAPL (5-minute intervals)
# This represents the "Historical Data" the engine will run against.
# REPLACE WITH REAL DATA FROM TASK 1
def generate_mock_data():
    dates = pd.date_range(start="2023-10-27 09:30:00", end="2023-10-27 16:00:00", freq="5T")
    n = len(dates)
    
    # Random walk for price
    price = 150.0 + np.cumsum(np.random.randn(n) * 0.2)
    
    # U-shaped volume profile (typical market behavior: high open/close, low midday)
    x = np.linspace(-1, 1, n)
    volume_profile = 1000 + 4000 * (x**2) + np.random.randint(0, 500, n)
    
    df = pd.DataFrame({
        'Open': price,
        'High': price + 0.1,
        'Low': price - 0.1,
        'Close': price + np.random.randn(n) * 0.05,
        'Volume': volume_profile.astype(int)
    }, index=dates)
    
    return df

# Load the mock data
market_data = generate_mock_data()
print("Market Data Loaded (First 5 rows):")
display(market_data.head())

Market Data Loaded (First 5 rows):


,Open,High,Low,Close,Volume
2023-10-27 09:30:00,150.024179,150.124179,149.924179,149.983540,5316
2023-10-27 09:35:00,149.832230,149.932230,149.732230,149.797410,5086
2023-10-27 09:40:00,149.870744,149.970744,149.770744,149.947920,5016
2023-10-27 09:45:00,149.934585,150.034585,149.834585,149.982637,4536
2023-10-27 09:50:00,150.226246,150.326246,150.126246,150.187002,4500


In [4]:
# ==========================================
# TASK 2 - CORE DATA STRUCTURES
# ==========================================
# Defining the fundamental objects required for the Execution Engine.

@dataclass
class ParentOrder:
    """
    Represents the large institutional trade to be executed.
    """
    ticker: str
    side: Literal['BUY', 'SELL']
    quantity: int
    start_time: pd.Timestamp
    end_time: pd.Timestamp

@dataclass
class ChildOrder:
    """
    Represents a slice of the parent order scheduled for a specific time.
    """
    timestamp: pd.Timestamp
    quantity: int
    
@dataclass
class ExecutionLog:
    """
    Records the actual result of a child order execution.
    """
    timestamp: pd.Timestamp
    requested_qty: int
    filled_qty: int
    execution_price: float
    market_volume: int
    strategy_name: str

In [5]:
# ==========================================
# TASK 2 - CORE EXECUTION ENGINE
# ==========================================
# This is the "Simulator" described in the proposal. 
# It takes a schedule of child orders and executes them against historical data.

class ExecutionEngine:
    def __init__(self, market_data: pd.DataFrame):
        self.market_data = market_data

    def validate_window(self, order: ParentOrder):
        """Checks if data exists for the requested window."""
        window_data = self.market_data.loc[order.start_time:order.end_time]
        if window_data.empty:
            raise ValueError(f"No market data found between {order.start_time} and {order.end_time}")
        return window_data

    def run(self, order: ParentOrder, strategy_schedule: List[ChildOrder], strategy_name: str) -> List[ExecutionLog]:
        """
        Simulates the execution of the child orders.
        
        Logic:
        1. Iterate through the strategy schedule.
        2. Match the child order timestamp to the market data candle.
        3. Execute at the 'Close' price (Phase 2.3 simplified model).
        4. Log the result.
        """
        logs = []
        
        # Filter data to the relevant window for efficiency
        # In a real engine, we might step through time index by index.
        # Here we map scheduled orders to their data points.
        
        for child in strategy_schedule:
            # Locate the specific candle for this child order
            if child.timestamp not in self.market_data.index:
                print(f"Warning: No market data for {child.timestamp}. Skipping slice.")
                continue
                
            candle = self.market_data.loc[child.timestamp]
            
            # EXECUTION MODEL (Phase 2.3)
            # Assumption: Market Order -> Fills immediately at Candle Close
            # Assumption: Full Fill (ignoring liquidity constraints for this version)
            
            fill_price = candle['Close']
            filled_qty = child.quantity # Assuming 100% fill rate for Core v1
            
            log = ExecutionLog(
                timestamp=child.timestamp,
                requested_qty=child.quantity,
                filled_qty=filled_qty,
                execution_price=fill_price,
                market_volume=int(candle['Volume']),
                strategy_name=strategy_name
            )
            logs.append(log)
            
        return logs

print("Execution Engine Class Initialized.")

Execution Engine Class Initialized.


In [6]:
# ==========================================
# TASK 2 - STRATEGY BASE & TWAP
# ==========================================
# Implementation of the base strategy interface and the 
# baseline TWAP (Time Weighted Average Price) strategy.

class Strategy:
    def generate_schedule(self, order: ParentOrder, market_data: pd.DataFrame) -> List[ChildOrder]:
        raise NotImplementedError("Strategies must implement generate_schedule")

class TWAPStrategy(Strategy):
    """
    Time Weighted Average Price Strategy.
    Splits the order quantity evenly across all available time bars in the window.
    """
    def generate_schedule(self, order: ParentOrder, market_data: pd.DataFrame) -> List[ChildOrder]:
        # 1. Identify valid timestamps in the window
        window_data = market_data.loc[order.start_time:order.end_time]
        timestamps = window_data.index.tolist()
        
        if not timestamps:
            return []
        
        num_slices = len(timestamps)
        
        # 2. Calculate slice size (Total Qty / N)
        # Use integer division, handle remainder on the last slice
        base_slice_size = order.quantity // num_slices
        remainder = order.quantity % num_slices
        
        schedule = []
        
        for i, ts in enumerate(timestamps):
            qty = base_slice_size
            # Add remainder to the last slice to ensure full fill
            if i == num_slices - 1:
                qty += remainder
                
            schedule.append(ChildOrder(timestamp=ts, quantity=qty))
            
        return schedule

print("TWAP Strategy Implementation Ready.")

TWAP Strategy Implementation Ready.


In [ ]:
# ==========================================
# TASK 4 - VWAP STRATEGY
# ==========================================
# Implementation of the Volume Weighted Average Price strategy.
# It uses the volume profile of the window to weight the slice sizes.

class VWAPStrategy(Strategy):
    """
    Volume Weighted Average Price Strategy.
    Allocates trade quantity proportional to the volume traded in each candle.
    
    Note: In a live trading scenario, this would use a 'forecast' volume curve.
    For this backtesting engine (ExecuSim), we use the actual historical volume
    (Perfect Foresight) to benchmark how well we COULD have done matching the market.
    """
    def generate_schedule(self, order: ParentOrder, market_data: pd.DataFrame) -> List[ChildOrder]:
        # 1. Identify valid timestamps and retrieve volume data
        window_data = market_data.loc[order.start_time:order.end_time]
        
        if window_data.empty:
            return []
        
        total_window_volume = window_data['Volume'].sum()
        
        # Guard against zero volume window
        if total_window_volume == 0:
            return TWAPStrategy().generate_schedule(order, market_data)
        
        schedule = []
        cumulative_qty = 0
        timestamps = window_data.index.tolist()
        volumes = window_data['Volume'].tolist()
        
        # 2. Iterate and calculate proportional slices
        for i, (ts, vol) in enumerate(zip(timestamps, volumes)):
            # Formula: Slice_i = Total_Order * (Vol_i / Total_Window_Vol)
            weight = vol / total_window_volume
            
            # Calculate tentative quantity
            if i == len(timestamps) - 1:
                # Last slice takes whatever is left to ensure exact quantity match
                qty = order.quantity - cumulative_qty
            else:
                qty = int(order.quantity * weight)
            
            cumulative_qty += qty
            schedule.append(ChildOrder(timestamp=ts, quantity=qty))
            
        return schedule

print("VWAP Strategy Implementation Ready.")

VWAP Strategy Implementation Ready.


In [8]:
# ==========================================
# EXECUTION DEMO (Testing the System)
# ==========================================
# This cell stitches Task 2 and Task 4 together to verify functionality.

# 1. Define the Parent Order
# Buy 50,000 shares of AAPL between 10:00 and 12:00
order = ParentOrder(
    ticker="AAPL",
    side="BUY",
    quantity=50000,
    start_time=pd.Timestamp("2023-10-27 10:00:00"),
    end_time=pd.Timestamp("2023-10-27 12:00:00")
)

# 2. Initialize Engine
engine = ExecutionEngine(market_data)

# 3. Run TWAP Simulation
twap_strat = TWAPStrategy()
twap_schedule = twap_strat.generate_schedule(order, market_data)
twap_logs = engine.run(order, twap_schedule, "TWAP")

# 4. Run VWAP Simulation
vwap_strat = VWAPStrategy()
vwap_schedule = vwap_strat.generate_schedule(order, market_data)
vwap_logs = engine.run(order, vwap_schedule, "VWAP")

# 5. Display Results (Simple DataFrame view of logs)
print(f"--- Simulation Results for {order.quantity} shares ---")

df_twap = pd.DataFrame([vars(l) for l in twap_logs])
df_vwap = pd.DataFrame([vars(l) for l in vwap_logs])

print("\n--- TWAP Execution (First 5 Slices) ---")
# Notice filled_qty is roughly constant
display(df_twap.head()) 

print("\n--- VWAP Execution (First 5 Slices) ---")
# Notice filled_qty varies with market_volume
display(df_vwap.head())

print("\n--- Sanity Check ---")
print(f"Total TWAP Executed: {df_twap['filled_qty'].sum()}")
print(f"Total VWAP Executed: {df_vwap['filled_qty'].sum()}")

--- Simulation Results for 50000 shares ---

--- TWAP Execution (First 5 Slices) ---


,timestamp,requested_qty,filled_qty,execution_price,market_volume,strategy_name
0,2023-10-27 10:00:00,2000,2000,150.083519,4073,TWAP
1,2023-10-27 10:05:00,2000,2000,150.017342,3807,TWAP
2,2023-10-27 10:10:00,2000,2000,149.924653,3654,TWAP
3,2023-10-27 10:15:00,2000,2000,149.946772,3684,TWAP
4,2023-10-27 10:20:00,2000,2000,150.305601,3566,TWAP



--- VWAP Execution (First 5 Slices) ---


,timestamp,requested_qty,filled_qty,execution_price,market_volume,strategy_name
0,2023-10-27 10:00:00,3191,3191,150.083519,4073,VWAP
1,2023-10-27 10:05:00,2983,2983,150.017342,3807,VWAP
2,2023-10-27 10:10:00,2863,2863,149.924653,3654,VWAP
3,2023-10-27 10:15:00,2886,2886,149.946772,3684,VWAP
4,2023-10-27 10:20:00,2794,2794,150.305601,3566,VWAP



--- Sanity Check ---
Total TWAP Executed: 50000
Total VWAP Executed: 50000
